# ML Phase 8: Model Training

This phase trains several regression models using the prepared Walmart sales data.

The models are trained on historical observations and evaluated using the future testing period created during chronological splitting.

The main evaluation metrics are:

- **MAE:** Average absolute prediction error.
- **RMSE:** Penalises large prediction errors more strongly.
- **R²:** Measures the proportion of sales variation explained by the model.

The goal is to train suitable regression models and prepare their results for comparison in the next phase.

In [1]:
# Importing the required libraries

from pathlib import Path
import pickle

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

## Importing Libraries

The required libraries are imported for:

- locating project files;
- loading the prepared datasets;
- creating regression models;
- generating predictions;
- calculating evaluation metrics;
- saving trained models using Pickle.

In [2]:
# Confirming the current notebook location

current_path = Path.cwd()

print("Current folder:", current_path)

Current folder: c:\Users\Udoch\walmart-sales-regression\notebooks


In [3]:
# Finding the project root safely

current_path = Path.cwd()

project_root = current_path

while not (project_root / "data").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError(
            "Could not locate the project root containing the data folder."
        )

    project_root = project_root.parent

print("Project root:", project_root)

Project root: c:\Users\Udoch\walmart-sales-regression


This approach is safer because it searches upwards until it finds your main data folder.

In [4]:
# Defining the required paths

prepared_data_path = project_root / "data" / "prepared"
models_path = project_root / "models"
results_path = project_root / "data" / "prepared"

models_path.mkdir(
    parents=True,
    exist_ok=True,
)

X_train_path = (
    prepared_data_path
    / "X_train_prepared.csv"
)

X_test_path = (
    prepared_data_path
    / "X_test_prepared.csv"
)

y_train_path = (
    prepared_data_path
    / "y_train.csv"
)

y_test_path = (
    prepared_data_path
    / "y_test.csv"
)

print("Prepared data folder:", prepared_data_path)
print("Models folder:", models_path)

Prepared data folder: c:\Users\Udoch\walmart-sales-regression\data\prepared
Models folder: c:\Users\Udoch\walmart-sales-regression\models


In [5]:
# Validating the input files

required_files = [
    X_train_path,
    X_test_path,
    y_train_path,
    y_test_path,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}"
        )

print("All prepared datasets were found successfully.")

All prepared datasets were found successfully.


In [6]:
# Loading the prepared datasets

X_train = pd.read_csv(
    X_train_path
)

X_test = pd.read_csv(
    X_test_path
)

y_train_df = pd.read_csv(
    y_train_path
)

y_test_df = pd.read_csv(
    y_test_path
)

print("Prepared datasets loaded successfully.")

Prepared datasets loaded successfully.


In [7]:
# Converting the target into one-dimensional Series

y_train = y_train_df.squeeze(
    axis="columns"
)

y_test = y_test_df.squeeze(
    axis="columns"
)

print("y_train type:", type(y_train))
print("y_test type:", type(y_test))

y_train type: <class 'pandas.Series'>
y_test type: <class 'pandas.Series'>


This makes the target format suitable for scikit-learn.

In [8]:
# Checking the dataset shapes

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5130, 69)
X_test shape: (1305, 69)
y_train shape: (5130,)
y_test shape: (1305,)


The number of rows in X_train must equal the number of values in y_train.

The number of rows in X_test must equal the number of values in y_test.

In [9]:
# Validating the matching rows

assert len(X_train) == len(y_train), (
    "X_train and y_train contain different numbers of rows."
)

assert len(X_test) == len(y_test), (
    "X_test and y_test contain different numbers of rows."
)

print("Feature and target row counts match.")

Feature and target row counts match.


In [10]:
# Checking for missing values

print(
    "Missing values in X_train:",
    X_train.isna().sum().sum(),
)

print(
    "Missing values in X_test:",
    X_test.isna().sum().sum(),
)

print(
    "Missing values in y_train:",
    y_train.isna().sum(),
)

print(
    "Missing values in y_test:",
    y_test.isna().sum(),
)

Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0


## Training Multiple Regression Models

Several regression algorithms will be trained to compare different modelling approaches.

The selected models are:

1. **Linear Regression** — provides a simple baseline model.
2. **Decision Tree Regressor** — learns non-linear decision rules.
3. **Random Forest Regressor** — combines multiple decision trees to improve prediction stability.
4. **Gradient Boosting Regressor** — builds trees sequentially to correct previous prediction errors.

All models are trained using the same training data and evaluated using the same testing data.

In [11]:
# Creating the models dictionary

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),
}

print("Models prepared:")
for model_name in models:
    print("-", model_name)

Models prepared:
- Linear Regression
- Decision Tree
- Random Forest
- Gradient Boosting


### Why use random_state=42?

It ensures that models containing random processes produce reproducible results.

### Why use n_jobs=-1?

For Random Forest, it allows the model to use all available processor cores.

In [12]:
# Creating the evaluation function

def evaluate_regression_model(
    model_name,
    actual_values,
    predicted_values,
):
    mae = mean_absolute_error(
        actual_values,
        predicted_values,
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_values,
            predicted_values,
        )
    )

    r2 = r2_score(
        actual_values,
        predicted_values,
    )

    result = {
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "r2_score": r2,
    }

    return result

## Evaluation Function

The evaluation function calculates three regression metrics:

- **MAE** measures the average absolute difference between actual and predicted weekly sales.
- **RMSE** measures prediction error while applying a greater penalty to large mistakes.
- **R² score** measures how much of the variation in weekly sales is explained by the model.

For MAE and RMSE, lower values indicate better performance.

For R², a value closer to 1 indicates better performance.

In [13]:
# Creating containers for the results

model_results = []
trained_models = {}
model_predictions = {}

These objects will store:

- the evaluation result for each model;
- the fitted model objects;
- the test predictions.

In [14]:
# Training all models

for model_name, model in models.items():

    print(f"Training {model_name}...")

    model.fit(
        X_train,
        y_train,
    )

    predictions = model.predict(
        X_test
    )

    result = evaluate_regression_model(
        model_name,
        y_test,
        predictions,
    )

    model_results.append(
        result
    )

    trained_models[model_name] = model

    model_predictions[model_name] = predictions

    print(f"{model_name} completed.")
    print()

Training Linear Regression...
Linear Regression completed.

Training Decision Tree...
Decision Tree completed.

Training Random Forest...
Random Forest completed.

Training Gradient Boosting...
Gradient Boosting completed.



In [15]:
# Converting the results into a DataFrame

results_df = pd.DataFrame(
    model_results
)

results_df

,model,mae,rmse,r2_score
0,Linear Regression,69989.550167,101451.251870,0.963850
1,Decision Tree,138187.230107,197760.411629,0.862636
2,Random Forest,80398.992144,116209.227572,0.952567
3,Gradient Boosting,176368.383347,201032.573717,0.858052


In [16]:
# Formatting the results

formatted_results_df = results_df.copy()

formatted_results_df["mae"] = (
    formatted_results_df["mae"]
    .round(2)
)

formatted_results_df["rmse"] = (
    formatted_results_df["rmse"]
    .round(2)
)

formatted_results_df["r2_score"] = (
    formatted_results_df["r2_score"]
    .round(4)
)

formatted_results_df

,model,mae,rmse,r2_score
0,Linear Regression,69989.55,101451.25,0.9638
1,Decision Tree,138187.23,197760.41,0.8626
2,Random Forest,80398.99,116209.23,0.9526
3,Gradient Boosting,176368.38,201032.57,0.8581


## Formatted Model Evaluation Results

The evaluation metrics were formatted to improve readability by rounding the values.

- **MAE** and **RMSE** were rounded to **2 decimal places**.
- **R² Score** was rounded to **4 decimal places**.

Formatting does **not** change the model performance; it only makes the results easier to read and compare.

### Model Performance Comparison

| Model | MAE | RMSE | R² Score |
|-------|------------:|------------:|---------:|
| Linear Regression | 69,989.55 | 101,451.25 | 0.9638 |
| Decision Tree | 138,187.23 | 197,760.41 | 0.8626 |
| Random Forest | 80,398.99 | 116,209.23 | 0.9526 |
| Gradient Boosting | 176,368.38 | 201,032.57 | 0.8581 |

### Interpretation

The formatted evaluation results make it easier to compare the performance of the four regression models.

**Linear Regression** achieved the best overall performance. It produced the **lowest MAE** (69,989.55), the **lowest RMSE** (101,451.25), and the **highest R² score** (0.9638). This indicates that the model predicted weekly sales with the smallest average error while explaining approximately **96.38%** of the variation in weekly sales.

**Random Forest** was the second-best performing model. Although it achieved a high R² score of **0.9526**, its MAE and RMSE were both higher than those of Linear Regression, indicating slightly larger prediction errors.

**Decision Tree** performed noticeably worse than the first two models. Its higher MAE and RMSE suggest less accurate predictions, while its R² score of **0.8626** shows that it explained less of the variation in weekly sales.

**Gradient Boosting** produced the weakest performance among the four models. It recorded the highest MAE and RMSE and the lowest R² score, indicating that its predictions were the least accurate using the default model settings.

### Model Ranking

Based on all three evaluation metrics, the models are ranked as follows:

1. Linear Regression
2. Random Forest
3. Decision Tree
4. Gradient Boosting

### Conclusion

At this stage, **Linear Regression** is the preferred model because it consistently achieved the lowest prediction errors and the highest explanatory power on the test dataset.

The next phase will further compare the models using visualisations and additional analysis before selecting the final model for deployment.

In [17]:
# Sorting the models by MAE

results_by_mae = (
    results_df
    .sort_values(
        by="mae",
        ascending=True,
    )
    .reset_index(
        drop=True
    )
)

results_by_mae

,model,mae,rmse,r2_score
0,Linear Regression,69989.550167,101451.251870,0.963850
1,Random Forest,80398.992144,116209.227572,0.952567
2,Decision Tree,138187.230107,197760.411629,0.862636
3,Gradient Boosting,176368.383347,201032.573717,0.858052


In [18]:
# Identifying the current best model

current_best_model_name = (
    results_by_mae
    .iloc[0]["model"]
)

current_best_model = trained_models[
    current_best_model_name
]

print(
    "Current best model by MAE:",
    current_best_model_name,
)

Current best model by MAE: Linear Regression


## Identifying the Current Best Model

The evaluation results were sorted by **Mean Absolute Error (MAE)** to identify the model with the smallest average prediction error.

The output shows that:

> **Current best model by MAE: Linear Regression**

This means that **Linear Regression** achieved the **lowest MAE** among all the models evaluated, making it the most accurate model based on average prediction error.

Since a lower MAE indicates better prediction accuracy, Linear Regression is currently the leading candidate for the final model.

This result is consistent with the previous evaluation, where Linear Regression also achieved:

- **Lowest MAE**
- **Lowest RMSE**
- **Highest R² Score**

The agreement across all three evaluation metrics provides strong evidence that Linear Regression performs better than the other models on this dataset.

At this stage, **Linear Regression** is selected as the current best-performing model and will be used in the next phase for final model selection, model serialization, and deployment.

In [19]:
# Displaying a sample of predictions

best_predictions = model_predictions[
    current_best_model_name
]

prediction_comparison = pd.DataFrame(
    {
        "actual_weekly_sales": y_test.reset_index(
            drop=True
        ),
        "predicted_weekly_sales": best_predictions,
    }
)

prediction_comparison["prediction_error"] = (
    prediction_comparison["actual_weekly_sales"]
    - prediction_comparison["predicted_weekly_sales"]
)

prediction_comparison.head(10)

,actual_weekly_sales,predicted_weekly_sales,prediction_error
0,1621031.70,1.574383e+06,46648.284904
1,1935869.10,1.975746e+06,-39877.227607
2,420789.74,4.275459e+05,-6756.141797
3,2105301.39,2.148867e+06,-43565.425023
4,351832.03,3.403018e+05,11530.252043
5,1616394.45,1.604321e+06,12073.610149
6,517420.23,5.766293e+05,-59209.100272
7,909989.45,9.315236e+05,-21534.184077
8,578539.86,5.727617e+05,5778.166256
9,1974687.51,1.949038e+06,25649.168346


## Sample Prediction Results

The table compares the **actual weekly sales** with the **predicted weekly sales** generated by the selected **Linear Regression** model.

Three columns are displayed:

- **actual_weekly_sales** – the true weekly sales recorded in the dataset.
- **predicted_weekly_sales** – the sales values predicted by the Linear Regression model.
- **prediction_error** – the difference between the actual and predicted values.

The prediction error is calculated as:

**Prediction Error = Actual Weekly Sales − Predicted Weekly Sales**

### Interpreting the Prediction Error

- A **positive** prediction error means the model **underestimated** the weekly sales because the actual value is greater than the predicted value.
- A **negative** prediction error means the model **overestimated** the weekly sales because the predicted value is greater than the actual value.
- A prediction error that is **close to zero** indicates that the prediction is very accurate.

### Interpretation of the Sample Output

The first ten predictions show that the model produces values that are generally close to the actual weekly sales.

For example:

- **Row 0:** The actual sales are **1,621,031.70**, while the model predicted approximately **1,574,383.42**, giving an error of about **46,648.28**. The model slightly underestimated the sales.
- **Row 1:** The model predicted a value slightly higher than the actual sales, resulting in a negative prediction error of approximately **−39,877.23**.
- Several other rows have prediction errors that are relatively small compared to the overall sales values, indicating that the model is producing reasonable predictions.

Overall, the prediction errors fluctuate between positive and negative values, suggesting that the model does not consistently overestimate or underestimate weekly sales. Instead, the errors are balanced around zero, which is a desirable characteristic of a well-performing regression model.

These sample predictions provide additional evidence that the selected **Linear Regression** model generalizes well to unseen data and is suitable for predicting Walmart weekly sales.

In [20]:
# Saving the evaluation results

model_results_path = (
    prepared_data_path
    / "model_training_results.csv"
)

results_df.to_csv(
    model_results_path,
    index=False,
)

print(
    "Model results saved to:",
    model_results_path,
)

Model results saved to: c:\Users\Udoch\walmart-sales-regression\data\prepared\model_training_results.csv


In [21]:
# Saveing all trained models using Pickle

trained_models_path = (
    models_path
    / "trained_regression_models.pkl"
)

with open(
    trained_models_path,
    "wb",
) as file:
    pickle.dump(
        trained_models,
        file,
    )

print(
    "Trained models saved to:",
    trained_models_path,
)

Trained models saved to: c:\Users\Udoch\walmart-sales-regression\models\trained_regression_models.pkl


This stores all four trained models inside one Pickle file.

In [22]:
# Saving the current best model separately

current_best_model_path = (
    models_path
    / "current_best_model.pkl"
)

with open(
    current_best_model_path,
    "wb",
) as file:
    pickle.dump(
        current_best_model,
        file,
    )

print(
    "Current best model saved to:",
    current_best_model_path,
)

Current best model saved to: c:\Users\Udoch\walmart-sales-regression\models\current_best_model.pkl


In [23]:
# Saving the predictions

predictions_df = pd.DataFrame(
    model_predictions
)

predictions_df.insert(
    0,
    "actual_weekly_sales",
    y_test.reset_index(
        drop=True
    ),
)

predictions_path = (
    prepared_data_path
    / "model_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False,
)

print(
    "Predictions saved to:",
    predictions_path,
)

Predictions saved to: c:\Users\Udoch\walmart-sales-regression\data\prepared\model_predictions.csv


In [24]:
# Testing that the Pickle file can be loaded

with open(
    trained_models_path,
    "rb",
) as file:
    loaded_models = pickle.load(
        file
    )

print(
    "Loaded models:",
    list(loaded_models.keys()),
)

Loaded models: ['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting']


## Phase 8 Summary

The prepared Walmart sales datasets were successfully loaded and validated.

Four regression models were trained:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

Each model generated predictions for the chronologically separated testing period.

Model performance was evaluated using:

- Mean Absolute Error
- Root Mean Squared Error
- R² Score

The trained models, evaluation results and predictions were saved for use during the model-comparison phase.

The model with the lowest MAE was identified as the current leading model. However, the final model will only be selected after considering MAE, RMSE, R², overfitting risk and prediction behaviour together.